In [ ]:
!fuser -k 5000/tcp

In [1]:
!pip install ultralytics easyocr pyttsx3 gTTS flask flask-cors pyngrok -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 14.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 r

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ===== AEGIS Full Pipeline - Imports =====
import os
import cv2
import torch
import base64
import shutil
import numpy as np
from ultralytics import YOLO
import easyocr
from gtts import gTTS
from IPython.display import Audio, display
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading
from pyngrok import ngrok
import requests
import json as pyjson
import socket

print("GPU available:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
GPU available: False


In [4]:
NGROK_AUTH_TOKEN = "ENTER YOUR NGROK AUTENTICATION TOKEN"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [5]:
PROJECT_ROOT = '/content/drive/MyDrive/AEGIS-AI'

model = YOLO('yolov8n.pt')  # or your fine-tuned checkpoint path
ocr_reader = easyocr.Reader(['en'], gpu=True)

print("Models loaded.")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteModels loaded.


In [6]:
KNOWN_WIDTHS = {
    'person': 0.5, 'chair': 0.5, 'car': 1.8, 'door': 0.9,
    'bench': 1.2, 'bicycle': 0.6, 'backpack': 0.3, 'bottle': 0.08
}
FOCAL_LENGTH = 700  # rough default; calibrate later if time allows

def estimate_distance(box_width_px, cls_name):
    real_width = KNOWN_WIDTHS.get(cls_name, 0.5)
    if box_width_px == 0:
        return None
    return round((real_width * FOCAL_LENGTH) / box_width_px, 1)

def get_direction(cx, frame_width):
    if cx < frame_width / 3:
        return "left"
    elif cx > 2 * frame_width / 3:
        return "right"
    return "ahead"

In [7]:
app = Flask(__name__)
CORS(app)

def decode_image(image_b64):
    img_bytes = base64.b64decode(image_b64.split(',')[1])
    np_arr = np.frombuffer(img_bytes, np.uint8)
    return cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

@app.route('/detect', methods=['POST'])
def detect():
    frame = decode_image(request.json['image'])
    results = model(frame, verbose=False)[0]
    h, w = results.orig_shape
    alerts = []
    boxes_out = []
    urgent = None
    primary_category = 'none'
    primary_steps = None

    for box in results.boxes:
        conf = float(box.conf[0])
        if conf < 0.4:
            continue
        cls_name = model.names[int(box.cls[0])]
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        box_w = x2 - x1
        cx = (x1 + x2) / 2
        direction = get_direction(cx, w)
        dist_m = estimate_distance(box_w, cls_name)
        steps = meters_to_steps(dist_m)
        dist_str = f"{steps} steps" if steps else ""
        alerts.append(f"{cls_name} {direction} {dist_str}".strip())

        guidance = get_guidance(direction, steps)
        if guidance and not urgent:
            urgent = f"{cls_name} {guidance}"

        if primary_category == 'none':
            primary_category = cls_name
            primary_steps = steps

        boxes_out.append({
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
            'label': f"{cls_name} {steps}steps" if steps else cls_name,
            'conf': conf
        })

    message = urgent if urgent else (", ".join(alerts[:2]) if alerts else "path clear")
    return jsonify({
        'message': message,
        'urgent': bool(urgent),
        'boxes': boxes_out,
        'frame_w': w, 'frame_h': h,
        'primary_category': primary_category,
        'primary_steps': primary_steps
    })

@app.route('/read_text', methods=['POST'])
def read_text():
    frame = decode_image(request.json['image'])
    temp_path = '/content/temp_ocr.jpg'
    cv2.imwrite(temp_path, frame)
    results = ocr_reader.readtext(temp_path)
    texts = [t for (bbox, t, conf) in results if conf > 0.4]
    message = " ".join(texts) if texts else "No text detected"
    return jsonify({'message': message})

@app.route('/find_object', methods=['POST'])
def find_object_route():
    frame = decode_image(request.json['image'])
    target = request.json.get('target', '').lower().strip()
    results = model(frame, verbose=False)[0]
    h, w = results.orig_shape

    for box in results.boxes:
        cls_name = model.names[int(box.cls[0])]
        conf = float(box.conf[0])
        if cls_name.lower() == target and conf > 0.35:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2
            direction = get_direction(cx, w)
            steps = meters_to_steps(estimate_distance(x2 - x1, cls_name))
            return jsonify({'message': f"{target} found {direction}, {steps} steps"})

    return jsonify({'message': f"{target} not found"})

print("Flask routes loaded.")

Flask routes loaded.


In [8]:
OPENROUTER_API_KEY = "ENTER YOUR API KEY FROM OPENROUTER"

def meters_to_steps(m):
    if m is None:
        return None
    return max(1, round(m / 0.75))

def get_guidance(direction, steps):
    if steps is not None and steps <= 1:
        if direction == "left":
            return "stop, turn right"
        elif direction == "right":
            return "stop, turn left"
        else:
            return "stop, obstacle very close"
    return None

@app.route('/parse_intent', methods=['POST'])
def parse_intent_route():
    user_text = request.json.get('text', '')

    prompt = f"""You are an intent parser for a blind navigation assistant.
Given a spoken command, respond ONLY with JSON, no other text.

Modes: "detect" (obstacle alerts), "read_text" (read signs/labels), "find_object" (locate a specific object)

Command: "{user_text}"

Respond in this exact format:
{{"mode": "find_object", "target": "microwave"}}

If mode is not find_object, set target to null.
"""

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": "meta-llama/llama-3.1-8b-instruct:free",
            "messages": [{"role": "user", "content": prompt}]
        }
    )

    try:
        content = response.json()['choices'][0]['message']['content']
        content = content.strip().strip('```json').strip('```').strip()
        parsed = pyjson.loads(content)
    except Exception as e:
        print("Intent parse error:", e)
        parsed = {"mode": "detect", "target": None}

    return jsonify(parsed)

print("Intent parsing + steps/guidance helpers ready.")

Intent parsing + steps/guidance helpers ready.


In [9]:
def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

if not is_port_in_use(5000):
    def run_flask():
        app.run(host='0.0.0.0', port=5000)
    flask_thread = threading.Thread(target=run_flask)
    flask_thread.start()
    print("Flask started fresh.")
else:
    print("Flask already running on port 5000 — not starting a new instance.")

# Only reconnect ngrok if you don't already have a tunnel open
public_url = ngrok.connect(5000)
print("Your public backend URL:", public_url)

Flask started fresh.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Your public backend URL: NgrokTunnel: "https://f306-34-122-34-87.ngrok-free.app" -> "http://localhost:5000"
